# Notebook 01 — Data Cleaning, Target Construction & Train/Test Split

**Project:** Boston Marathon BQ Predictor  
**Author:** Gian Marco  
**Date:** April 2026

## Objectives

1. Load the 3 CSV files (Results, Races, BQStandards)
2. Apply cleaning criteria documented in `DECISIONS.md`
3. Build the target variable `BQ`
4. Merge with race metadata (Races.csv)
5. Generate a stratified sample of 300k rows
6. Apply temporal split: train 2022-2023 / test 2024
7. Save processed datasets

---
## 1. Imports and configuration

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Routes
RAW_DATA_DIR = Path('../data/raw')
PROCESSED_DATA_DIR = Path('../data/processed')
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Config pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

---
## 2. Load the 3 datasets

In [3]:
results = pd.read_csv(RAW_DATA_DIR / 'Results.csv', low_memory=False)
races = pd.read_csv(RAW_DATA_DIR / 'Races.csv')
bq = pd.read_csv(RAW_DATA_DIR / 'BQStandards.csv')

print(f'Results: {results.shape}')
print(f'Races:   {races.shape}')
print(f'BQ:      {bq.shape}')

Results: (1760979, 12)
Races:   (762, 8)
BQ:      (22, 3)


In [4]:
# Vista rápida
results.head(3)

,Year,Race,Name,Country,Zip,City,State,Gender,Age,Finish,Overall Place,Gender Place
0,2022,NYC Marathon,Evans Chebet,US,NaN,NaN,New York,M,33,7721.0,1,1.0
1,2022,NYC Marathon,Shura Kitata,US,NaN,NaN,New York,M,26,7734.0,2,2.0
2,2022,NYC Marathon,Abdi Nageeye,US,NaN,NaN,New York,M,33,7831.0,3,3.0


In [5]:
races.head(3)

,Year,Date,Race,City,State,Finishers,Include,Category
0,2022,2022-11-06,NYC Marathon,New York,NY,47738,Yes,NaN
1,2023,2023-11-05,NYC Marathon,New York,NY,51348,Yes,NaN
2,2022,2022-10-30,Marine Corps Marathon,Washington,DC,11257,Yes,NaN


In [6]:
bq.head(3)

,Gender,Age Bracket,Standard
0,M,Under 35,10800
1,M,35-39,11100
2,M,40-44,11400


---
## 3. Results.csv cleaning

We apply the filters documented in `DECISIONS.md` (Decision 6).  
We track how many records are removed at each step to ensure full traceability.

In [7]:
df = results.copy()
cleaning_log = [('Initial', len(df))]

# 3.1 Convert Age to numeric (some values contained blank spaces)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# 3.2 Remove DNF (Finish = 0)
df = df[df['Finish'] > 0]
cleaning_log.append(('After removing DNF', len(df)))

# 3.3 Keep only M and F (BQ Standards do not define other genders)
df = df[df['Gender'].isin(['M', 'F'])]
cleaning_log.append(('After filtering M/F', len(df)))

# 3.4 Keep reasonable ages (18-85)
df = df[(df['Age'] >= 18) & (df['Age'] <= 85)]
cleaning_log.append(('After filtering valid age range', len(df)))

# 3.5 Drop columns with many nulls or no predictive value
df = df.drop(columns=['Zip', 'City', 'State', 'Name'])
cleaning_log.append(('After dropping irrelevant columns', len(df)))

# Visual log
print('RESULTS CLEANING LOG')
print('=' * 50)
for step, n in cleaning_log:
    print(f'{step:40s} {n:>12,}')

RESULTS CLEANING LOG
Initial                                     1,760,979
After removing DNF                          1,746,679
After filtering M/F                         1,722,190
After filtering valid age range             1,615,216
After dropping irrelevant columns           1,615,216


### 3.6 Country normalization

We detected duplicates in country codes (`US`/`USA`, `GB`/`GBR`, `DE`/`GER`).  
These were standardized to ensure consistency.

In [8]:
# Exhaustive mapping: ALL non-ISO-2 codes → ISO 2-letter codes
country_mapping = {
    # Standard ISO 3-letter codes (common)
    'USA': 'US', 'GBR': 'GB', 'CAN': 'CA', 'FRA': 'FR', 'AUS': 'AU',
    'GER': 'DE', 'ITA': 'IT', 'IRL': 'IE', 'MEX': 'MX', 'RUS': 'RU',
    'ESP': 'ES', 'JPN': 'JP', 'ARG': 'AR', 'TWN': 'TW', 'DEN': 'DK',
    'SWE': 'SE', 'AUT': 'AT', 'BRA': 'BR', 'SIN': 'SG', 'BEL': 'BE',
    'HKG': 'HK', 'POL': 'PL', 'NZL': 'NZ', 'NOR': 'NO', 'SVK': 'SK',
    'POR': 'PT', 'CZE': 'CZ', 'IND': 'IN', 'EST': 'EE', 'FIN': 'FI',
    'CHN': 'CN', 'MAR': 'MA', 'TUR': 'TR', 'ISR': 'IL', 'PHI': 'PH',
    'THA': 'TH', 'COL': 'CO', 'ROU': 'RO', 'HUN': 'HU', 'KEN': 'KE',
    'LTU': 'LT', 'ISL': 'IS', 'UKR': 'UA', 'VIE': 'VN', 'KOR': 'KR',
    'PER': 'PE', 'VEN': 'VE', 'PAN': 'PA', 'TAN': 'TZ', 'ECU': 'EC',
    'DOM': 'DO', 'CRO': 'HR', 'ETH': 'ET', 'LUX': 'LU', 'EGY': 'EG',
    'UGA': 'UG', 'SRB': 'RS', 'BOL': 'BO',

    # Less common ISO 3-letter codes (countries with few runners)
    'MGL': 'MN', 'ERI': 'ER', 'GRE': 'GR', 'JOR': 'JO', 'CRC': 'CR',
    'IRI': 'IR', 'TRI': 'TT', 'KAZ': 'KZ', 'PUR': 'PR', 'LIE': 'LI',
    'ALG': 'DZ', 'LIB': 'LB', 'PAK': 'PK', 'BAN': 'BD', 'ZIM': 'ZW',
    'GEO': 'GE', 'GHA': 'GH', 'ANG': 'AO', 'AZE': 'AZ', 'NEP': 'NP',
    'BAH': 'BS', 'NCA': 'NI', 'TPE': 'TW', 'ARM': 'AM', 'KSA': 'SA',
    'KGZ': 'KG', 'LBA': 'LY', 'MLT': 'MT', 'MDA': 'MD', 'MON': 'MC',
    'SEN': 'SN', 'SRI': 'LK', 'TOG': 'TG', 'TRN': 'TT', 'TUN': 'TN',
    'BDI': 'BI', 'COD': 'CD', 'MAC': 'MO', 'CIV': 'CI', 'CAF': 'CF',
    'VIN': 'VC', 'MOZ': 'MZ', 'COM': 'KM', 'DMA': 'DM', 'DJI': 'DJ',

    # Non-standard / atypical codes
    'NED': 'NL',  # Netherlands
    'SUI': 'CH',  # Switzerland
    'INA': 'ID',  # Indonesia
    'RSA': 'ZA',  # South Africa
    'CHI': 'CL',  # Chile (note: NOT China, which is CN)
    'BUR': 'UNKNOWN',  # Ambiguous: Burundi or Burkina Faso (only 2 cases)

    # Extra
    'LAT': 'LV', 'BRN': 'BH', 'MAS': 'MY', 'BUL': 'BG', 'FRO': 'FO',
    'PAR': 'PY', 'QAT': 'QA', 'SWZ': 'SZ', 'GUA': 'GT', 'KOS': 'XK',
    'ESA': 'SV', 'SLO': 'SI', 'BLR': 'BY', 'CYP': 'CY', 'BIH': 'BA',
    'MDV': 'MV', 'URU': 'UY', 'HON': 'HN', 'GIB': 'GI', 'MKD': 'MK',
    'ZAM': 'ZM', 'GRL': 'GL', 'SUD': 'SD', 'CHA': 'TD', 'BRU': 'BN',
    'AND': 'AD', 'IRQ': 'IQ', 'ALB': 'AL', 'MRI': 'MU', 'PLE': 'PS',
    'BER': 'BM', 'GUM': 'GU', 'NGR': 'NG', 'JAM': 'JM', 'CUB': 'CU',
    'GRN': 'GD', 'AFG': 'AF', 'HAI': 'HT', 'SUR': 'SR', 'LCA': 'LC',
    'NAM': 'NA', 'UZB': 'UZ', 'SYR': 'SY', 'MNE': 'ME', 'CMR': 'CM',
    'OMN': 'OM', 'TKM': 'TM', 'BEN': 'BJ', 'SMR': 'SM', 'PNG': 'PG',
    'KUW': 'KW', 'CPV': 'CV', 'BIZ': 'BZ', 'MAD': 'MG', 'GUY': 'GY',
    'UAE': 'AE', 'CAM': 'KH', 'BOT': 'BW', 'LAO': 'LA', 'PLW': 'PW',
    'RWA': 'RW', 'MAW': 'MW',
}

df['Country'] = df['Country'].replace(country_mapping)
df['Country'] = df['Country'].fillna('UNKNOWN')

# Final verification
countries_3_letters = df[df['Country'].str.len() == 3]['Country'].unique()
if len(countries_3_letters) > 0:
    print(f'WARNING: 3-letter country codes still remain: {countries_3_letters}')
    print(f'Total affected runners: {df[df["Country"].isin(countries_3_letters)].shape[0]}')
else:
    print('OK: all country codes are ISO-2 or UNKNOWN')

print(f'\nUnique countries after normalization: {df["Country"].nunique()}')
print(f'\nTop 15 countries:')
print(df['Country'].value_counts().head(15))

OK: all country codes are ISO-2 or UNKNOWN

Unique countries after normalization: 216

Top 15 countries:
Country
US         775536
GB         159970
ZA          56242
CA          54653
AU          50958
DE          47887
IT          42237
NL          37351
MX          36064
IE          35325
ES          31211
RU          28116
UNKNOWN     20562
JP          20159
FR          19220
Name: count, dtype: int64


---
## 4. Merge with Races.csv

We want to include race-level information:  
- `Category` (difficulty)  
- `Finishers` (race size)  

We also filter only `Include == 'Yes'` to follow the dataset author's selection criteria.

⚠️ **Important note:**  
Before applying the `Include=Yes` filter, we store a snapshot with all races.  
This allows us to later build a narrative slice for Spain (Spanish races are not marked as `Include=Yes` because the dataset focuses mainly on USA/Canada + London/Berlin).

In [9]:
# Save snapshot with all races (pre-Include filter) for the Spain slice
df_all_races = df.copy()

# Now apply filter for the main pipeline
races_clean = races[races['Include'] == 'Yes'].copy()
races_clean = races_clean[['Year', 'Race', 'City', 'State', 'Finishers', 'Category']]
races_clean = races_clean.rename(columns={'City': 'Race_City', 'State': 'Race_State'})

n_before = len(df)
df = df.merge(races_clean, on=['Year', 'Race'], how='inner')

print(f'Rows after merge (Include=Yes races only): {len(df):,}')
print(f'Rows removed: {n_before - len(df):,}')

cleaning_log.append(('After merge with Races (Include=Yes)', len(df)))

Rows after merge (Include=Yes races only): 1,068,226
Rows removed: 546,990


---
## 5. Target construction (`BQ`)

Each runner is assigned to an age group according to the Boston Marathon standards.  
We then merge this information and create the binary target variable.

In [10]:
def assign_age_bracket(age):
    """Assign age bracket according to official Boston Marathon categories."""
    if age < 35: return 'Under 35'
    elif age < 40: return '35-39'
    elif age < 45: return '40-44'
    elif age < 50: return '45-49'
    elif age < 55: return '50-54'
    elif age < 60: return '55-59'
    elif age < 65: return '60-64'
    elif age < 70: return '65-69'
    elif age < 75: return '70-74'
    elif age < 80: return '75-79'
    else: return '80 and Over'

df['Age Bracket'] = df['Age'].apply(assign_age_bracket)

# Merge with BQ standards to obtain qualifying time
df = df.merge(bq, on=['Gender', 'Age Bracket'], how='left')

# Create target variable
df['es_BQ'] = (df['Finish'] <= df['Standard']).astype(int)

print('Target distribution:')
print(df['es_BQ'].value_counts())
print(f'\n% BQ: {df["es_BQ"].mean()*100:.2f}%')
print(f'Imbalance ratio: 1:{(df["es_BQ"]==0).sum() / (df["es_BQ"]==1).sum():.2f}')

Target distribution:
es_BQ
0    922271
1    145955
Name: count, dtype: int64

% BQ: 13.66%
Imbalance ratio: 1:6.32


---
## 6. Sanity checks

Before sampling, we verify that the data is consistent and makes sense.

In [11]:
print('% BQ by year (should increase based on previous observations):')
print(df.groupby('Year')['es_BQ'].agg(['mean', 'sum', 'count']).round(3))

print('\n% BQ by gender:')
print(df.groupby('Gender')['es_BQ'].agg(['mean', 'count']).round(3))

print('\nRace distribution (top 10):')
print(df['Race'].value_counts().head(10))

% BQ by year (should increase based on previous observations):
       mean    sum   count
Year                      
2022  0.117  33659  286548
2023  0.144  74276  515888
2024  0.143  38020  265790

% BQ by gender:
         mean   count
Gender               
F       0.141  424447
M       0.134  643779

Race distribution (top 10):
Race
London Marathon          143144
NYC Marathon              98934
Chicago Marathon          87211
Berlin Marathon           77742
Boston Marathon           52099
LA Marathon               32743
Honolulu Marathon         28548
Disney World Marathon     25402
Marine Corps Marathon     24757
Philadelphia Marathon     19672
Name: count, dtype: int64


---
## 7. Stratified sampling

**Strategy:** Sample 300k rows stratified by `BQ × Year × Gender`  
to preserve key distributions (see `DECISIONS.md`, Decision 4).

In [12]:
TARGET_SAMPLE_SIZE = 300_000

# Compute sampling fraction
frac = TARGET_SAMPLE_SIZE / len(df)
print(f'Sampling fraction: {frac:.4f}')

# Stratification across multiple combined columns
strata = df['es_BQ'].astype(str) + '_' + df['Year'].astype(str) + '_' + df['Gender']

df_sample = (
    df.groupby(strata, group_keys=False)
      .apply(lambda g: g.sample(frac=frac, random_state=RANDOM_STATE))
      .reset_index(drop=True)
)

print(f'Sample size: {len(df_sample):,}')
print(f'\nBQ rate in sample: {df_sample["es_BQ"].mean()*100:.2f}% (target: ~12.76%)')

print('\nBQ rate by year in sample (should resemble original distribution):')
print(df_sample.groupby('Year')['es_BQ'].agg(['mean', 'count']).round(3))

Sampling fraction: 0.2808
Sample size: 300,000

BQ rate in sample: 13.66% (target: ~12.76%)

BQ rate by year in sample (should resemble original distribution):
       mean   count
Year               
2022  0.117   80475
2023  0.144  144881
2024  0.143   74644


---
## 8. Temporal split

**Train:** 2022-2023  
**Test:** 2024  

Full justification is provided in `DECISIONS.md` (Decision 3).

In [13]:
train = df_sample[df_sample['Year'].isin([2022, 2023])].reset_index(drop=True)
test = df_sample[df_sample['Year'] == 2024].reset_index(drop=True)

print(f'Train: {len(train):,} rows  |  BQ rate: {train["es_BQ"].mean()*100:.2f}%')
print(f'Test:  {len(test):,} rows  |  BQ rate: {test["es_BQ"].mean()*100:.2f}%')
print(f'\nTrain/Test ratio: {len(train)/len(test):.2f}')

Train: 225,356 rows  |  BQ rate: 13.45%
Test:  74,644 rows  |  BQ rate: 14.30%

Train/Test ratio: 3.02


---
## 9. Narrative slice: Spain

We reserve Spanish race records for later analysis (Notebook 09)  
and for the Streamlit demo.

In [14]:
# Apply the same target construction process to df_all_races (pre-Include filter)
df_all_races['Age Bracket'] = df_all_races['Age'].apply(assign_age_bracket)
df_all_races = df_all_races.merge(bq, on=['Gender', 'Age Bracket'], how='left')
df_all_races['es_BQ'] = (df_all_races['Finish'] <= df_all_races['Standard']).astype(int)

# Identify races held in Spain
spain_keywords = ['Madrid', 'Barcelona', 'Sevilla', 'Valencia', 'Lanzarote']
spain_races_mask = df_all_races['Race'].str.contains('|'.join(spain_keywords), case=False, na=False)

# 'Zurich Marato' (Catalan/Spanish naming for Barcelona/Sevilla)
zurich_spain = df_all_races['Race'].str.contains('Zurich Marato', case=False, na=False)
spain_races_mask = spain_races_mask | zurich_spain

spain_slice = df_all_races[spain_races_mask].reset_index(drop=True)

print(f'Spain slice: {len(spain_slice):,} rows')
print(f'\nSpanish races included:')
print(spain_slice.groupby(['Race', 'Year']).agg(
    finishers=('es_BQ', 'count'),
    bq_pct=('es_BQ', 'mean')
).round(3))

Spain slice: 31,419 rows

Spanish races included:
                              finishers  bq_pct
Race                    Year                   
RnR Madrid Marathon     2023       7387   0.107
                        2024       9059   0.107
Zurich Marato Barcelona 2023       7917   0.162
Zurich Maraton Sevilla  2024       7056   0.316


---
## 10. Save processed datasets

In [15]:
# Define specific paths
TRAIN_DIR = Path('../data/train')
TEST_DIR = Path('../data/test')

# Save cleaned datasets into their respective folders
train.to_csv(TRAIN_DIR / 'train.csv', index=False)
test.to_csv(TEST_DIR / 'test.csv', index=False)
spain_slice.to_csv(TEST_DIR / 'spain_slice.csv', index=False)

# cleaning_log contains pipeline metadata → save in processed
cleaning_df = pd.DataFrame(cleaning_log, columns=['Step', 'Row_count'])
cleaning_df.to_csv(PROCESSED_DATA_DIR / 'cleaning_log.csv', index=False)

print('Files saved:')
print(f'\ndata/train/:')
for f in sorted(TRAIN_DIR.iterdir()):
    if f.name != '.gitkeep':
        size_mb = f.stat().st_size / 1024**2
        print(f'  {f.name:25s} {size_mb:>6.1f} MB')

print(f'\ndata/test/:')
for f in sorted(TEST_DIR.iterdir()):
    if f.name != '.gitkeep':
        size_mb = f.stat().st_size / 1024**2
        print(f'  {f.name:25s} {size_mb:>6.1f} MB')

print(f'\ndata/processed/:')
for f in sorted(PROCESSED_DATA_DIR.iterdir()):
    if f.name != '.gitkeep':
        size_mb = f.stat().st_size / 1024**2
        print(f'  {f.name:25s} {size_mb:>6.1f} MB')

Files saved:

data/train/:
  train.csv                   19.0 MB
  train_features.csv          16.9 MB

data/test/:
  spain_slice.csv              2.1 MB
  test.csv                     6.3 MB
  test_features.csv            5.6 MB

data/processed/:
  advanced_results.csv         0.0 MB
  baseline_results.csv         0.0 MB
  cleaning_log.csv             0.0 MB
  finishers_with_clusters.parquet    4.9 MB
  imbalance_results.csv        0.0 MB
  slice_edad.csv               0.0 MB
  slice_genero.csv             0.0 MB
  slice_pais.csv               0.0 MB
  threshold_tuning_results.csv    0.0 MB


---
## Notebook Summary

| Step | Result |
|---|---|
| Raw data | 1.76M rows |
| After cleaning | ~1.6M rows |
| After merge with Races (Include=Yes) | ~1.5M rows (estimated) |
| Stratified sample | 300k rows |
| Train (2022-2023) | ~265k rows |
| Test (2024) | ~35k rows |
| Spain slice | ~6k rows |
| **BQ target %** | **~12.76%** |